In [17]:
# 필요한 패키지
# pip install sqlalchemy pymysql pandas

import pandas as pd
from sqlalchemy import create_engine, text

# ── DB 접속 정보 ───────────────────────────────────────────────────────────────
from DATA.stock_invest_function import get_db_host

db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

TABLE_NAME = "Korea_company_valuation_ver2"  # 스키마: investar.TABLE_NAME

def make_engine(db):
    url = (
        f"mysql+pymysql://{db['user']}:{db['password']}"
        f"@{db['host']}:{db['port']}/{db['database']}?charset=utf8mb4"
    )
    return create_engine(url, pool_pre_ping=True, future=True)

engine = make_engine(db_info)

# 1) forecast_date의 unique 값 추출
def get_unique_forecast_dates(include_null=False):
    q = f"""
        SELECT DISTINCT forecast_date
        FROM {TABLE_NAME}
        {"WHERE forecast_date IS NOT NULL" if not include_null else ""}
        ORDER BY forecast_date
    """
    with engine.begin() as conn:
        df = pd.read_sql(q, conn, parse_dates=["forecast_date"])
    return df["forecast_date"]

# 2) (ticker, forecast_date, keyword)로 indicator에 keyword가 포함된 값 조회 + date 기준 정렬
#    여러 indicator가 매칭되면 행으로 반환(롱 포맷). wide=True면 indicator별 칼럼으로 피벗.
def get_series_by_keyword(ticker, forecast_date, keyword, wide=False):
    sql = text(f"""
        SELECT `date`, `ticker`, `indicator`, `value`, `forecast_date`
        FROM {TABLE_NAME}
        WHERE ticker = :ticker
          AND forecast_date = :fdate
          AND indicator LIKE :kw
        ORDER BY `date`
    """)
    with engine.begin() as conn:
        df = pd.read_sql(
            sql, conn,
            params={"ticker": ticker, "fdate": forecast_date, "kw": f"%{keyword}%"},
            parse_dates=["date", "forecast_date"]
        )
    # 숫자형 보정
    if not df.empty:
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
    if wide and not df.empty:
        df_wide = df.pivot_table(index="date", columns="indicator", values="value", aggfunc="last").sort_index()
        df_wide = df_wide.rename_axis(None, axis=1)
        return df_wide
    return df  # 롱 포맷: date, indicator, value …

# 3) indicator의 unique 값 추출
def get_unique_indicators(keyword=None):
    cond = "" if not keyword else "WHERE indicator LIKE :kw"
    sql = text(f"SELECT DISTINCT indicator FROM {TABLE_NAME} {cond} ORDER BY indicator")
    with engine.begin() as conn:
        df = pd.read_sql(sql, conn, params=(None if not keyword else {"kw": f"%{keyword}%"}))
    return df["indicator"]

# 4) (ticker, indicator, forecast_date 두 개) 입력 시 두 기간 차이 비교
#    반환: date 기준 병합(outer), col: value_fd1, value_fd2, diff = fd2 - fd1
def compare_indicator_between_dates(ticker, indicator, forecast_date_1, forecast_date_2):
    base_sql = text(f"""
        SELECT `date`, `value`
        FROM {TABLE_NAME}
        WHERE ticker = :ticker
          AND indicator = :indicator
          AND forecast_date = :fdate
        ORDER BY `date`
    """)
    with engine.begin() as conn:
        df1 = pd.read_sql(
            base_sql, conn,
            params={"ticker": ticker, "indicator": indicator, "fdate": forecast_date_1},
            parse_dates=["date"]
        )
        df2 = pd.read_sql(
            base_sql, conn,
            params={"ticker": ticker, "indicator": indicator, "fdate": forecast_date_2},
            parse_dates=["date"]
        )

    # 숫자형 보정
    for d in (df1, df2):
        if not d.empty:
            d["value"] = pd.to_numeric(d["value"], errors="coerce")

    df1 = df1.rename(columns={"value": f"value_{pd.to_datetime(forecast_date_1).date()}"})
    df2 = df2.rename(columns={"value": f"value_{pd.to_datetime(forecast_date_2).date()}"})

    out = pd.merge(df1, df2, on="date", how="outer").sort_values("date").set_index("date")
    if out.shape[1] == 2:
        cols = out.columns.tolist()
        out["diff"] = out[cols[1]] - out[cols[0]]  # fd2 - fd1
    return out

# ── 사용 예시 ─────────────────────────────────────────────────────────────────
# if __name__ == "__main__":
#     # 1) forecast_date 목록
#     print(get_unique_forecast_dates().tail())
#
#     # 2) 키워드로 조회 (롱/와이드)
#     ex_long = get_series_by_keyword(ticker="A005930", forecast_date="2025-10-26", keyword="revenue", wide=False)
#     ex_wide = get_series_by_keyword(ticker="A005930", forecast_date="2025-10-26", keyword="revenue", wide=True)
#     print(ex_long.head())
#     print(ex_wide.head())
#
#     # 3) indicator 유니크
#     print(get_unique_indicators().head())
#     # 특정 키워드만
#     print(get_unique_indicators(keyword="forecast").head())
#
#     # 4) 두 forecast_date 비교
#     comp = compare_indicator_between_dates(
#         ticker="A005930",
#         indicator="revenue_ensemble_forecast",   # 예: 정확한 indicator 이름 입력
#         forecast_date_1="2025-10-26",
#         forecast_date_2="2025-10-29"
#     )
#     print(comp.tail())



In [18]:
print(get_unique_forecast_dates().tail())

0   2025-10-26
1   2025-10-29
2   2025-10-30
Name: forecast_date, dtype: datetime64[ns]


In [34]:
ex_long = get_series_by_keyword(ticker="A035720", forecast_date="2025-10-30", keyword="mc", wide=False)

In [35]:
ex_long

,date,ticker,indicator,value,forecast_date
0,2025-11-30,A035720,mc_sarima_noexog,4.095405e+10,2025-10-30
1,2025-11-30,A035720,mc_ets,4.001312e+10,2025-10-30
2,2025-11-30,A035720,mc_prophet,3.458355e+10,2025-10-30
3,2025-11-30,A035720,mc_lstm,3.472820e+10,2025-10-30
4,2025-11-30,A035720,mc_theta,4.041644e+10,2025-10-30
...,...,...,...,...,...
60,2026-11-30,A035720,mc_sarima_noexog,3.928010e+10,2025-10-30
61,2026-11-30,A035720,mc_ets,3.700832e+10,2025-10-30
62,2026-11-30,A035720,mc_prophet,3.436496e+10,2025-10-30
63,2026-11-30,A035720,mc_lstm,4.364178e+10,2025-10-30


In [36]:
# ex_long → indicator를 컬럼으로 피벗
ex_pivot = (
    ex_long
    .pivot_table(
        index=["date", "ticker"],      # 행 인덱스
        columns="indicator",           # 열로 변환할 컬럼
        values="value",                # 값으로 쓸 컬럼
        aggfunc="last"                 # 중복 시 마지막 값 사용
    )
    .reset_index()                     # date, ticker를 일반 컬럼으로 되돌림
)

# 필요 시 indicator 컬럼명 정리 (MultiIndex 제거)
ex_pivot.columns.name = None

# 확인
print(ex_pivot.head())


        date   ticker        mc_ets       mc_lstm    mc_prophet  \
0 2025-11-30  A035720  4.001312e+10  3.472820e+10  3.458355e+10   
1 2025-12-31  A035720  3.912271e+10  3.706790e+10  3.459376e+10   
2 2026-01-31  A035720  3.832224e+10  3.803692e+10  3.358941e+10   
3 2026-02-28  A035720  3.739503e+10  3.862360e+10  3.289588e+10   
4 2026-03-31  A035720  3.893277e+10  4.059235e+10  3.532792e+10   

   mc_sarima_noexog      mc_theta  
0      4.095405e+10  4.041644e+10  
1      4.208958e+10  4.065142e+10  
2      4.023904e+10  4.052697e+10  
3      3.999001e+10  4.040251e+10  
4      4.115247e+10  4.093638e+10  


In [37]:
ex_pivot.tail(14)

,date,ticker,mc_ets,mc_lstm,mc_prophet,mc_sarima_noexog,mc_theta
0,2025-11-30,A035720,4.001312e+10,3.472820e+10,3.458355e+10,4.095405e+10,4.041644e+10
1,2025-12-31,A035720,3.912271e+10,3.706790e+10,3.459376e+10,4.208958e+10,4.065142e+10
2,2026-01-31,A035720,3.832224e+10,3.803692e+10,3.358941e+10,4.023904e+10,4.052697e+10
3,2026-02-28,A035720,3.739503e+10,3.862360e+10,3.289588e+10,3.999001e+10,4.040251e+10
4,2026-03-31,A035720,3.893277e+10,4.059235e+10,3.532792e+10,4.115247e+10,4.093638e+10
5,2026-04-30,A035720,4.048504e+10,4.070932e+10,3.651123e+10,4.309487e+10,4.080989e+10
6,2026-05-31,A035720,4.097850e+10,4.066849e+10,3.810884e+10,4.338239e+10,4.068340e+10
7,2026-06-30,A035720,4.575166e+10,4.159396e+10,4.129667e+10,4.827884e+10,4.108083e+10
8,2026-07-31,A035720,4.650068e+10,4.186278e+10,4.128661e+10,4.896934e+10,4.095270e+10
9,2026-08-31,A035720,4.443050e+10,4.203474e+10,3.976627e+10,4.687678e+10,4.082458e+10


In [32]:
psr_long = get_series_by_keyword(ticker="A035720", forecast_date="2025-10-30", keyword="rev", wide=False)

# ex_long → indicator를 컬럼으로 피벗
psr_pivot = (
    psr_long
    .pivot_table(
        index=["date", "ticker"],      # 행 인덱스
        columns="indicator",           # 열로 변환할 컬럼
        values="value",                # 값으로 쓸 컬럼
        aggfunc="last"                 # 중복 시 마지막 값 사용
    )
    .reset_index()                     # date, ticker를 일반 컬럼으로 되돌림
)

# 필요 시 indicator 컬럼명 정리 (MultiIndex 제거)
psr_pivot.columns.name = None

# 확인
print(psr_pivot.head())

        date   ticker  revenue_ets  revenue_ets_ttm  revenue_lstm  \
0 2004-03-31  A035720   44068105.0              NaN    44068105.0   
1 2004-06-30  A035720   46937151.0              NaN    46937151.0   
2 2004-09-30  A035720   49452370.0              NaN    49452370.0   
3 2004-12-31  A035720   42933918.0      183391544.0    42933918.0   
4 2005-03-31  A035720   47540502.0      186863941.0    47540502.0   

   revenue_lstm_ttm  revenue_prophet  revenue_prophet_ttm  revenue_sarima  \
0               NaN       44068105.0                  NaN      44068105.0   
1               NaN       46937151.0                  NaN      46937151.0   
2               NaN       49452370.0                  NaN      49452370.0   
3       183391544.0       42933918.0          183391544.0      42933918.0   
4       186863941.0       47540502.0          186863941.0      47540502.0   

   revenue_sarima_exog  revenue_sarima_exog_ttm  revenue_sarima_ttm  \
0           44068105.0                      NaN    

In [33]:
psr_pivot.tail(14)

,date,ticker,revenue_ets,revenue_ets_ttm,revenue_lstm,revenue_lstm_ttm,revenue_prophet,revenue_prophet_ttm,revenue_sarima,revenue_sarima_exog,revenue_sarima_exog_ttm,revenue_sarima_ttm,revenue_theta,revenue_theta_ttm
77,2023-06-30,A035720,1.923255e+09,6.871747e+09,1.923255e+09,6.871747e+09,1.923255e+09,6.871747e+09,1.923255e+09,1.923255e+09,6.871747e+09,6.871747e+09,1.923255e+09,6.871747e+09
78,2023-09-30,A035720,2.011472e+09,7.024522e+09,2.011472e+09,7.024522e+09,2.011472e+09,7.024522e+09,2.011472e+09,2.011472e+09,7.024522e+09,7.024522e+09,2.011472e+09,7.024522e+09
79,2023-12-31,A035720,1.998506e+09,7.557002e+09,1.998506e+09,7.557002e+09,1.998506e+09,7.557002e+09,1.998506e+09,1.998506e+09,7.557002e+09,7.557002e+09,1.998506e+09,7.557002e+09
80,2024-03-31,A035720,1.988350e+09,7.921583e+09,1.988350e+09,7.921583e+09,1.988350e+09,7.921583e+09,1.988350e+09,1.988350e+09,7.921583e+09,7.921583e+09,1.988350e+09,7.921583e+09
81,2024-06-30,A035720,2.004898e+09,8.003226e+09,2.004898e+09,8.003226e+09,2.004898e+09,8.003226e+09,2.004898e+09,2.004898e+09,8.003226e+09,8.003226e+09,2.004898e+09,8.003226e+09
82,2024-09-30,A035720,1.921398e+09,7.913153e+09,1.921398e+09,7.913153e+09,1.921398e+09,7.913153e+09,1.921398e+09,1.921398e+09,7.913153e+09,7.913153e+09,1.921398e+09,7.913153e+09
83,2024-12-31,A035720,1.957046e+09,7.871692e+09,1.957046e+09,7.871692e+09,1.957046e+09,7.871692e+09,1.957046e+09,1.957046e+09,7.871692e+09,7.871692e+09,1.957046e+09,7.871692e+09
84,2025-03-31,A035720,1.863731e+09,7.747073e+09,1.863731e+09,7.747073e+09,1.863731e+09,7.747073e+09,1.863731e+09,1.863731e+09,7.747073e+09,7.747073e+09,1.863731e+09,7.747073e+09
85,2025-06-30,A035720,2.028313e+09,7.770488e+09,2.028313e+09,7.770488e+09,2.028313e+09,7.770488e+09,2.028313e+09,2.028313e+09,7.770488e+09,7.770488e+09,2.028313e+09,7.770488e+09
86,2025-09-30,A035720,2.055664e+09,7.904754e+09,2.101685e+09,7.950774e+09,2.207735e+09,8.056825e+09,2.183650e+09,NaN,NaN,8.032739e+09,2.076468e+09,7.925558e+09
